# Stage 4: Feature Engineering & Lag Generation

Now that our customer demographic data is thoroughly cleaned and imputed, we must build the historical context that our machine learning model requires. 

Because we are utilizing a gradient-boosted tree ensemble (XGBoost/LightGBM), the model treats each row as an independent event. It cannot inherently look backward in time across rows. To solve this, we will explicitly engineer **Lag Features**. 

### What We Are Building:
1. **Lag 1 Features (`_lag_1`)**: The exact product ownership status of the customer in the *previous* month ($t-1$). This tells the model what products the customer already holds.
2. **Lag 2 Features (`_lag_2`)**: The product ownership status from *two months prior* ($t-2$). This helps the model detect short-term ownership trajectories.
3. **Total Holdings Aggregate**: A derived column counting the absolute number of products a customer holds, allowing the engine to differentiate between highly active banking users and single-account users.

### Environment and Imports

In [1]:
# GLOBAL IMPORTS & ENVIRONMENT SETUP
import pandas as pd
import numpy as np
import warnings
import os


# Suppress downcasting and performance warnings from pandas chain-operations
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Set pandas display options for better engineering readability in the notebook
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

print("Global dependencies successfully loaded. Environment ready for pipeline operations.")

Global dependencies successfully loaded. Environment ready for pipeline operations.


## Load the Full Cohort (All 3 Months)

We load `cohort.parquet`  NOT the targets file.

The cohort file contains all 48 columns (24 customer features + 24 product 
ownership flags) across all 3 snapshot months for ~600k active customers.

We need all 3 months present simultaneously so we can shift product ownership 
backward in time to create lag features. The targets file only has 3 columns 
and has already lost the historical product states  it cannot be used here.

In [2]:
# Load the full 3-month cohort this file has all 24 product columns
COHORT_PATH = "../data/processed/cohort.parquet"

df_cohort = pd.read_parquet(COHORT_PATH)
# Reset index to ensure a clean slate for any future operations that might rely on row positions
df_cohort = df_cohort.reset_index(drop=True)

print(f"Cohort loaded successfully.")
print(f"Shape: {df_cohort.shape}")
print(f"Columns (first 10): {list(df_cohort.columns[:10])}")
print(f"Months present: {sorted(df_cohort['fecha_dato'].unique())}")
print(f"Unique customers: {df_cohort['ncodpers'].nunique():,}")

Cohort loaded successfully.
Shape: (2766687, 48)
Columns (first 10): ['fecha_dato', 'ncodpers', 'ind_empleado', 'pais_residencia', 'sexo', 'age', 'fecha_alta', 'ind_nuevo', 'antiguedad', 'indrel']
Months present: [Timestamp('2016-03-28 00:00:00'), Timestamp('2016-04-28 00:00:00'), Timestamp('2016-05-28 00:00:00')]
Unique customers: 922,229


##  Define the 24 Product Columns

These are the binary product ownership flags (0 = does not own, 1 = owns).

We define them dynamically from the actual cohort columns rather than 
hardcoding all 25 names this protects against any column that was 
absent from the cohort during notebook 02 selection.

`ind_gco_fin_ult1` (guarantee account) is confirmed absent from this 
dataset version and is excluded automatically by the dynamic selection.

In [3]:
# All known Santander product column names from the competition description
ALL_KNOWN_PRODUCT_COLS = [
    "ind_ahor_fin_ult1", "ind_aval_fin_ult1", "ind_cco_fin_ult1",
    "ind_cder_fin_ult1", "ind_cno_fin_ult1",  "ind_ctju_fin_ult1",
    "ind_ctma_fin_ult1", "ind_ctop_fin_ult1", "ind_ctpp_fin_ult1",
    "ind_deco_fin_ult1", "ind_deme_fin_ult1", "ind_dela_fin_ult1",
    "ind_ecue_fin_ult1", "ind_fond_fin_ult1", "ind_gco_fin_ult1",
    "ind_hip_fin_ult1",  "ind_plan_fin_ult1", "ind_pres_fin_ult1",
    "ind_reca_fin_ult1", "ind_tjcr_fin_ult1", "ind_valo_fin_ult1",
    "ind_viv_fin_ult1",  "ind_nomina_ult1",   "ind_nom_pens_ult1",
    "ind_recibo_ult1"
]

# Dynamically keep only columns that actually exist in the cohort
# This prevents KeyErrors in the lag generation loop
PRODUCT_COLS = [c for c in ALL_KNOWN_PRODUCT_COLS if c in df_cohort.columns]

print(f"Product columns in dataset : {len(PRODUCT_COLS)}")
print(f"Excluded (not in cohort)   : {[c for c in ALL_KNOWN_PRODUCT_COLS if c not in df_cohort.columns]}")
print(f"Final PRODUCT_COLS list    : {PRODUCT_COLS}")

Product columns in dataset : 24
Excluded (not in cohort)   : ['ind_gco_fin_ult1']
Final PRODUCT_COLS list    : ['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']


## Income (Renta) Imputation on Cohort

The full cohort has 659,619 null values in `renta` (gross income) 
and 11,962 nulls in `nomprov` (province) confirmed in our earlier 
null check.

We apply the two-tier strategy here because this is where the cohort 
data lives and where imputation has a real, lasting effect on the 
feature matrix that reaches the model.

- Tier 1: Known province → fill with that province's median income  
- Tier 2: Unknown province → fill with global cohort median

In [4]:
# Force renta to numeric some entries may be strings or empty
df_cohort['renta'] = pd.to_numeric(
    df_cohort['renta'].astype(str).str.strip(),
    errors='coerce'
)

# TIER 1: Province-level median from rows where both fields are present
province_median = (
    df_cohort
    .dropna(subset=['nomprov', 'renta'])
    .groupby('nomprov')['renta']
    .median()
)

# TIER 2: Global cohort median as fallback for unknown provinces
global_median_renta = df_cohort['renta'].median()

def impute_renta(row):
    """
    Two-tier renta imputation.
    Tier 1: use province median if province is known.
    Tier 2: use global cohort median if province is null or unknown.
    """
    if pd.isna(row['renta']):
        if pd.notna(row['nomprov']) and row['nomprov'] in province_median.index:
            return province_median[row['nomprov']]
        return global_median_renta
    return row['renta']

df_cohort['renta'] = df_cohort.apply(impute_renta, axis=1)

# Verify
remaining = df_cohort['renta'].isnull().sum()
print(f"Renta nulls after imputation : {remaining}")
print(f"Global fallback median       : {global_median_renta:,.0f}")
print(f"Province medians computed    : {len(province_median)}")
assert remaining == 0, "Imputation failed — nulls remain in renta"

Renta nulls after imputation : 0
Global fallback median       : 101,493
Province medians computed    : 52


## Sort Chronologically Within Each Customer Group

Lag generation works by shifting rows backward within each customer's 
timeline. For this shift to produce the correct historical state, rows 
must be in strict chronological order per customer before we shift.

We sort by `ncodpers` (customer ID) then `fecha_dato` (snapshot date).

In [5]:
# Sort the cohort so each customer's rows appear in time order
# This is mandatory groupby().shift() depends entirely on row order
df_cohort = df_cohort.sort_values(
    by=['ncodpers', 'fecha_dato']
).reset_index(drop=True)

print(f"Sorted by ncodpers + fecha_dato.")
print(f"Date range: {df_cohort['fecha_dato'].min()} → {df_cohort['fecha_dato'].max()}")
print(f"Shape unchanged: {df_cohort.shape}")

Sorted by ncodpers + fecha_dato.
Date range: 2016-03-28 00:00:00 → 2016-05-28 00:00:00
Shape unchanged: (2766687, 48)


## Generate Lag 1 and Lag 2 Product Features

For each of the 24 product columns we create two lag versions:

- **lag_1**: what did the customer own in the *previous* month (t-1)?  
- **lag_2**: what did the customer own *two months ago* (t-2)?

`groupby('ncodpers').shift(1)` moves each customer's product row 
down by one position within their group so the March value appears 
on the April row as lag_1, and the February value appears as lag_2.

This gives the model explicit historical context without it needing 
to remember across rows.

In [6]:
# Generate lag features for all 24 product columns
for prod in PRODUCT_COLS:
    # lag_1: previous month's product state for this customer
    df_cohort[f"{prod}_lag_1"] = (
        df_cohort.groupby('ncodpers')[prod].shift(1)
    )
    # lag_2: two months ago product state for this customer
    df_cohort[f"{prod}_lag_2"] = (
        df_cohort.groupby('ncodpers')[prod].shift(2)
    )

# Compute aggregate summary features from the lag columns
lag_1_cols = [f"{p}_lag_1" for p in PRODUCT_COLS]
lag_2_cols = [f"{p}_lag_2" for p in PRODUCT_COLS]

# total products held last month  captures overall banking engagement
df_cohort['total_products_held_lag_1'] = df_cohort[lag_1_cols].sum(axis=1)

# product velocity — positive = growing, negative = churning, 0 = stable
df_cohort['product_velocity'] = (
    df_cohort['total_products_held_lag_1'] -
    df_cohort[lag_2_cols].sum(axis=1)
)

# Validation: lag_1 columns should have real values for non-first-month rows
non_zero = df_cohort[lag_1_cols].sum().sum()
print(f"Lag Feature Validation:")
print(f"  Total non-zero entries across lag_1 columns: {non_zero:,}")
print(f"  (Should be >> 0 — if 0, sort order was wrong)")
print(f"  New columns added: {len(lag_1_cols) + len(lag_2_cols) + 2}")
print(f"  Current shape: {df_cohort.shape}")

Lag Feature Validation:
  Total non-zero entries across lag_1 columns: 2,461,582
  (Should be >> 0 — if 0, sort order was wrong)
  New columns added: 50
  Current shape: (2766687, 98)


## Filter to the Target Month and Drop Boundary Rows

The lag columns will contain NaN for a customer's earliest month(s) 
because there is no "previous month" to look back to. These rows 
cannot be used for training and must be dropped.

We then filter to **May 2016 only** this is the target month whose 
product additions we are trying to predict. March and April rows 
served their purpose as the source of lag values and are no longer needed.

In [7]:
# Filter to May 2016 the final snapshot month (our training target month)
TARGET_MONTH = '2016-05-28'
df_may = df_cohort[df_cohort['fecha_dato'] == TARGET_MONTH].copy()

print(f"Rows in May 2016 snapshot: {len(df_may):,}")

# Drop rows where lag columns are NaN (boundary rows with incomplete history)
initial_count = len(df_may)
df_may = df_may.dropna(subset=['total_products_held_lag_1',
                                'total_products_held_lag_2'] 
                        if 'total_products_held_lag_2' in df_may.columns
                        else ['total_products_held_lag_1'])
final_count = len(df_may)

print(f"Dropped {initial_count - final_count:,} boundary rows with incomplete lag history")
print(f"Rows remaining: {final_count:,}")

Rows in May 2016 snapshot: 922,229
Dropped 0 boundary rows with incomplete lag history
Rows remaining: 922,229


##  Merge Lag Features with Target Labels

The target labels (which product each customer added) live in 
`target.parquet` from Notebook 03. We merge on `ncodpers` to attach 
the target product index to each customer's May 2016 feature row.

We exclude the raw product ownership columns (PRODUCT_COLS) from the 
final merge the model should not see current ownership directly, 
only the lag versions. Showing current ownership would leak the answer.

In [8]:
# Load target labels from Notebook 03
TARGET_PATH = "../data/processed/target.parquet"
df_targets = pd.read_parquet(TARGET_PATH).reset_index(drop=True)

print(f"Targets loaded: {df_targets.shape}")

# Merge May features with target labels
# Keep all target rows (left join from targets) 
# some customers may have multiple target rows if they added multiple products
df_features = df_targets.merge(
    df_may.drop(columns=PRODUCT_COLS),  # drop raw product cols to prevent leakage
    on='ncodpers',
    how='left'
)

print(f"Feature matrix shape after merge: {df_features.shape}")
print(f"Null counts in key columns:")
print(df_features[['total_products_held_lag_1', 'product_velocity']].isnull().sum())

Targets loaded: (33870, 2)
Feature matrix shape after merge: (33870, 75)
Null counts in key columns:
total_products_held_lag_1    0
product_velocity             0
dtype: int64


## Step 7: Save Final Feature Matrix

Save the complete feature matrix to `data/processed/features.parquet`.
This is the input to Notebook 05 (data splitting and DMatrix formatting).

In [9]:
# save the final feature matrix for modeling
os.makedirs("../data/processed", exist_ok=True)
FEATURES_PATH = "../data/processed/features.parquet"
# Use index=False to avoid saving the default integer index as a column in the parquet file
df_features.to_parquet(FEATURES_PATH, index=False)

print(f"Features saved to {FEATURES_PATH}")
print(f"Final shape: {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")

Features saved to ../data/processed/features.parquet
Final shape: (33870, 75)
Columns: ['ncodpers', 'target_product_idx', 'fecha_dato', 'ind_empleado', 'pais_residencia', 'sexo', 'age', 'fecha_alta', 'ind_nuevo', 'antiguedad', 'indrel', 'ult_fec_cli_1t', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'conyuemp', 'canal_entrada', 'indfall', 'tipodom', 'cod_prov', 'nomprov', 'ind_actividad_cliente', 'renta', 'segmento', 'ind_ahor_fin_ult1_lag_1', 'ind_ahor_fin_ult1_lag_2', 'ind_aval_fin_ult1_lag_1', 'ind_aval_fin_ult1_lag_2', 'ind_cco_fin_ult1_lag_1', 'ind_cco_fin_ult1_lag_2', 'ind_cder_fin_ult1_lag_1', 'ind_cder_fin_ult1_lag_2', 'ind_cno_fin_ult1_lag_1', 'ind_cno_fin_ult1_lag_2', 'ind_ctju_fin_ult1_lag_1', 'ind_ctju_fin_ult1_lag_2', 'ind_ctma_fin_ult1_lag_1', 'ind_ctma_fin_ult1_lag_2', 'ind_ctop_fin_ult1_lag_1', 'ind_ctop_fin_ult1_lag_2', 'ind_ctpp_fin_ult1_lag_1', 'ind_ctpp_fin_ult1_lag_2', 'ind_deco_fin_ult1_lag_1', 'ind_deco_fin_ult1_lag_2', 'ind_deme_fin_ult1_lag_1', 'ind_deme_f

## Final Column Audit Before Save

Confirm no near-empty or leakage columns survived the merge.
We check for columns that were flagged as >99% null in Notebook 03
and drop them here if present, documenting the reason.

In [10]:
# Columns confirmed >99% null in Notebook 03 must not enter the model
COLS_TO_DROP = []

# Check each known problem column
KNOWN_BAD = ['ult_fec_cli_1t', 'conyuemp']

for col in KNOWN_BAD:
    if col in df_features.columns:
        null_pct = df_features[col].isnull().mean() * 100
        print(f"{col}: {null_pct:.1f}% null — DROPPING")
        COLS_TO_DROP.append(col)
    else:
        print(f"{col}: not present — OK")

# Drop confirmed bad columns
if COLS_TO_DROP:
    df_features = df_features.drop(columns=COLS_TO_DROP)
    print(f"\nDropped: {COLS_TO_DROP}")

# Also drop fecha_dato it is a date identifier not a model feature
# The model cannot use a raw date string as a numeric feature
if 'fecha_dato' in df_features.columns:
    df_features = df_features.drop(columns=['fecha_dato'])
    print(f"Dropped: fecha_dato (date identifier, not a model feature)")

# Also drop fecha_alta raw join date, not useful as numeric input
if 'fecha_alta' in df_features.columns:
    df_features = df_features.drop(columns=['fecha_alta'])
    print(f"Dropped: fecha_alta (raw date string, not a model feature)")

print(f"\nFinal clean shape: {df_features.shape}")
print(f"Remaining columns: {list(df_features.columns[:10])}...")

ult_fec_cli_1t: 100.0% null — DROPPING
conyuemp: 100.0% null — DROPPING

Dropped: ['ult_fec_cli_1t', 'conyuemp']
Dropped: fecha_dato (date identifier, not a model feature)
Dropped: fecha_alta (raw date string, not a model feature)

Final clean shape: (33870, 71)
Remaining columns: ['ncodpers', 'target_product_idx', 'ind_empleado', 'pais_residencia', 'sexo', 'age', 'ind_nuevo', 'antiguedad', 'indrel', 'indrel_1mes']...


## Save Cleaned Feature Matrix

Save the final audited feature matrix. This is the definitive input 
to Notebook 05. All near-empty columns and date identifiers removed.

In [11]:
# save the final feature matrix for modeling
os.makedirs("../data/processed", exist_ok=True)
FEATURES_PATH = "../data/processed/features.parquet"

df_features.to_parquet(FEATURES_PATH, index=False)

print(f"Saved → {FEATURES_PATH}")
print(f"Final shape: {df_features.shape}")
print(f"All columns: {list(df_features.columns)}")

Saved → ../data/processed/features.parquet
Final shape: (33870, 71)
All columns: ['ncodpers', 'target_product_idx', 'ind_empleado', 'pais_residencia', 'sexo', 'age', 'ind_nuevo', 'antiguedad', 'indrel', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'canal_entrada', 'indfall', 'tipodom', 'cod_prov', 'nomprov', 'ind_actividad_cliente', 'renta', 'segmento', 'ind_ahor_fin_ult1_lag_1', 'ind_ahor_fin_ult1_lag_2', 'ind_aval_fin_ult1_lag_1', 'ind_aval_fin_ult1_lag_2', 'ind_cco_fin_ult1_lag_1', 'ind_cco_fin_ult1_lag_2', 'ind_cder_fin_ult1_lag_1', 'ind_cder_fin_ult1_lag_2', 'ind_cno_fin_ult1_lag_1', 'ind_cno_fin_ult1_lag_2', 'ind_ctju_fin_ult1_lag_1', 'ind_ctju_fin_ult1_lag_2', 'ind_ctma_fin_ult1_lag_1', 'ind_ctma_fin_ult1_lag_2', 'ind_ctop_fin_ult1_lag_1', 'ind_ctop_fin_ult1_lag_2', 'ind_ctpp_fin_ult1_lag_1', 'ind_ctpp_fin_ult1_lag_2', 'ind_deco_fin_ult1_lag_1', 'ind_deco_fin_ult1_lag_2', 'ind_deme_fin_ult1_lag_1', 'ind_deme_fin_ult1_lag_2', 'ind_dela_fin_ult1_lag_1', 'ind_dela_fin_ult1_la